# Minnie 1822 features and UMAP

Save morphology features and their UMAP embedding in CCM format.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import umap
from deltalake import write_deltalake
from sklearn.preprocessing import StandardScaler

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import read_embedding_locations, write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    EmbeddingLocation,
    EmbeddingMethod,
    EmbeddingSpace,
)

In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM minnie 1822"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "minnie65"
DATASET_ALL = "minnie65_v1822_all"
FEATURE_SET_ID = "minnie65_v1822_morphology_features"
EMBEDDING_SPACE_ID = "minnie65_v1822_morphology_umap"
HIERARCHY_ID = "minnie65_v1822_cell_types"
RANDOM_STATE = 42

## features

In [ ]:
feature_metadata = pd.read_csv(DATA_ROOT / "minnie_cell_features.csv")
source_features = pd.read_parquet(DATA_ROOT / "minnie_features.parquet")

metadata_columns = {
    "id", "description", "unit", "data_type", "range_min", "range_max"
}
assert metadata_columns <= set(feature_metadata.columns)
assert feature_metadata["id"].is_unique
feature_columns = feature_metadata["id"].tolist()
assert set(feature_columns) <= set(source_features.columns)
assert (source_features["root_id"] == source_features["pt_root_id_x"]).all()
assert (source_features["root_id"] == source_features["pt_root_id_y"]).all()

source_features = source_features.copy()
source_features["id"] = source_features["root_id"].astype(str)
duplicate_rows = source_features.loc[source_features["id"].duplicated(keep=False)]
for _, group in duplicate_rows.groupby("id"):
    assert group[feature_columns].nunique(dropna=False).le(1).all(), (
        "Duplicate root ID has conflicting feature values."
    )
source_features = source_features.drop_duplicates("id", keep="first").reset_index(drop=True)

numeric_features = source_features[feature_columns].astype("float64")
valid_rows = numeric_features.notna().all(axis=1) & np.isfinite(numeric_features).all(axis=1)
valid_features = source_features.loc[valid_rows, ["id", *feature_columns]].reset_index(drop=True)
assert valid_features["id"].is_unique
print(f"source rows: {len(source_features):,}")
print(f"valid UMAP rows: {len(valid_features):,}")

## umap

In [ ]:
feature_values = valid_features[feature_columns].to_numpy(dtype="float64")
scaled_values = StandardScaler().fit_transform(feature_values)
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=RANDOM_STATE,
)
coordinates = reducer.fit_transform(scaled_values)
assert coordinates.shape == (len(valid_features), 2)
assert np.isfinite(coordinates).all()
embedding_frame = pd.DataFrame(
    {
        "dataitem_id": valid_features["id"],
        "x": coordinates[:, 0].astype("float64"),
        "y": coordinates[:, 1].astype("float64"),
    }
)

## registered cells

In [ ]:
associations = pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association").filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("dataset_id") == DATASET_ALL)
)
assert associations.height > 0, (
    "No Minnie 1822 DataItems found; run etl_minnie1822_01.ipynb first."
)
registered_ids = set(associations["dataitem_id"].to_list())
reduced_features = valid_features.loc[
    valid_features["id"].isin(registered_ids)
].reset_index(drop=True)
reduced_embedding = embedding_frame.loc[
    embedding_frame["dataitem_id"].isin(registered_ids)
].reset_index(drop=True)

feature_ids = set(reduced_features["id"])
embedding_ids = set(reduced_embedding["dataitem_id"])
assert feature_ids == embedding_ids
assert feature_ids <= registered_ids
assert reduced_features["id"].is_unique
assert reduced_embedding["dataitem_id"].is_unique
print(f"registered Minnie 1822 cells: {len(registered_ids):,}")
print(f"feature and embedding rows retained: {len(feature_ids):,}")
print(f"valid feature rows excluded from CCM output: {len(valid_features) - len(feature_ids):,}")

## ccm features

In [ ]:
def optional_float(value):
    return None if pd.isna(value) else float(value)


feature_definitions = [
    CellFeatureDefinition(
        id=row.id,
        description=row.description,
        unit=row.unit,
        data_type=row.data_type,
        range_min=optional_float(row.range_min),
        range_max=optional_float(row.range_max),
        project_id=PROJECT_ID,
        feature_set_id=FEATURE_SET_ID,
    )
    for row in feature_metadata.itertuples(index=False)
]
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description="Minnie65 soma and dendrite morphology features used for UMAP.",
    feature_definition_ids=feature_columns,
    extraction_method="Cached minnie_features.parquet morphology pipeline",
    project_id=PROJECT_ID,
)
print(write_models(feature_definitions, output_root=OUTPUT_ROOT))
print(write_models([feature_set], output_root=OUTPUT_ROOT))

In [ ]:
feature_wide = reduced_features.copy()
feature_wide["project_id"] = PROJECT_ID
feature_wide["feature_set_id"] = FEATURE_SET_ID
feature_wide = feature_wide[["id", "project_id", "feature_set_id", *feature_columns]]
for definition in feature_definitions:
    feature_wide[definition.id] = feature_wide[definition.id].astype(
        np.dtype(definition.data_type)
    )

feature_schema = build_cell_feature_matrix_schema(
    feature_set, feature_definitions, cell_index_column="id"
)
feature_table = pa.Table.from_pandas(
    feature_wide, schema=feature_schema, preserve_index=False
)
feature_path = OUTPUT_ROOT / f"cellfeatures/{FEATURE_SET_ID}"
write_deltalake(
    feature_path,
    feature_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
feature_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{feature_path.resolve()}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([feature_matrix], output_root=OUTPUT_ROOT))

## ccm embedding

In [ ]:
embedding_space = EmbeddingSpace(
    id=EMBEDDING_SPACE_ID,
    name="Minnie65 morphology UMAP",
    description="Two-dimensional UMAP of Minnie65 morphology features.",
    embedding_method=EmbeddingMethod.UMAP,
    parameters_json=json.dumps(
        {
            "preprocessing": "StandardScaler",
            "n_components": 2,
            "n_neighbors": 15,
            "min_dist": 0.1,
            "metric": "euclidean",
            "random_state": RANDOM_STATE,
            "fit_rows": len(valid_features),
        }
    ),
    input_feature_set_id=FEATURE_SET_ID,
    input_features_description=(
        f"All {len(feature_columns)} metadata-declared features; UMAP was fit before "
        "restricting persisted coordinates to registered Minnie 1822 DataItems."
    ),
    project_id=PROJECT_ID,
)
embedding_locations = [
    EmbeddingLocation(
        dataitem_id=row.dataitem_id,
        embedding_space=EMBEDDING_SPACE_ID,
        x=float(row.x),
        y=float(row.y),
        project_id=PROJECT_ID,
    )
    for row in reduced_embedding.itertuples(index=False)
]
print(write_models([embedding_space], output_root=OUTPUT_ROOT))
print(write_models(embedding_locations, output_root=OUTPUT_ROOT))

## verify

In [ ]:
stored_features = pl.read_delta(feature_path).filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("feature_set_id") == FEATURE_SET_ID)
)
stored_embedding = read_embedding_locations(
    PROJECT_ID,
    embedding_space_ids=EMBEDDING_SPACE_ID,
    output_root=OUTPUT_ROOT,
)
stored_definitions = pl.read_delta(OUTPUT_ROOT / "cellfeaturedefinition").filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("feature_set_id") == FEATURE_SET_ID)
)
assert stored_features.height == len(feature_ids)
assert stored_embedding.height == len(feature_ids)
assert stored_definitions.height == len(feature_columns)
assert set(stored_features["id"].to_list()) == feature_ids
assert set(stored_embedding["dataitem_id"].to_list()) == feature_ids
assert set(stored_features.columns) == {
    "id", "project_id", "feature_set_id", *feature_columns
}
print("Minnie 1822 morphology features and UMAP embedding verified.")

## Plot UMAP by cell type

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from connects_common_connectivity.io import DatasetReader

reader = DatasetReader(OUTPUT_ROOT)
leaf_column = f"{HIERARCHY_ID}_level_2"
cell_types = (
    reader.read_dataset(
        DATASET_ALL,
        featuresets=[],
        clustersets=HIERARCHY_ID,
    )
    .select("dataitem_id", pl.col(leaf_column).alias("cell_type"))
)
cell_type_colors = (
    pl.read_delta(OUTPUT_ROOT / "cluster")
    .filter(
        (pl.col("hierarchy_id") == HIERARCHY_ID)
        & (pl.col("level") == 2)
    )
    .select(pl.col("id").alias("cell_type"), "hex_color")
)
plot_nodes = (
    read_embedding_locations(
        PROJECT_ID,
        embedding_space_ids=EMBEDDING_SPACE_ID,
        output_root=OUTPUT_ROOT,
    )
    .join(cell_types, on="dataitem_id", how="inner")
    .join(cell_type_colors, on="cell_type", how="inner")
)
assert not plot_nodes.is_empty()
assert plot_nodes["hex_color"].null_count() == 0
cell_type_palette = dict(
    plot_nodes.select("cell_type", "hex_color").unique().iter_rows()
)

figure, axis = plt.subplots(figsize=(8, 7))
sns.scatterplot(
    data=plot_nodes.to_pandas(),
    x="x",
    y="y",
    hue="cell_type",
    palette=cell_type_palette,
    s=7,
    linewidth=0,
    alpha=0.8,
    ax=axis,
)
axis.set(
    title="Minnie65 morphology UMAP",
    xlabel="UMAP 1",
    ylabel="UMAP 2",
)
axis.legend(title="Cell type", bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine(ax=axis)
figure.tight_layout()